# Task 4: Numerical Prediction (Regression) - Mental Health Mood Prediction

**Goal:** Predict next-day mood as a **regression** problem, treating mood as a continuous value (1-10 scale).

We apply two fundamentally different regression approaches:
1. **Gradient Boosting Regressor** - a powerful tabular method (non-temporal)
2. **LSTM Regressor** - a recurrent neural network that explicitly models temporal sequences

**Key difference from classification (Task 2A):** Instead of discretizing mood into classes (Low/Medium/High), we predict the exact continuous mood value. This preserves the ordinal structure of mood and allows finer-grained predictions, but requires different evaluation metrics (MSE, MAE, R-squared instead of accuracy/F1).

Both models are evaluated with a time-based train/test split to respect the temporal nature of the data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("All imports successful.")

## 1. Load Data

We load the feature-engineered dataset from Task 1C. Unlike the classification task, we keep the target as a **continuous** value - no discretization needed.

In [ ]:
# Load the feature-engineered dataset from Task 1C
df = pd.read_csv('../data/dataset_features.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")
df.head()

In [ ]:
# --- Identify key columns ---
# TODO: Adjust these column names to match your actual dataset_features.csv
# Expected columns: a patient/id column, a date/time column, feature columns, and a target mood column.

ID_COL = 'id'          # patient identifier column
DATE_COL = 'date'      # date column (for temporal ordering)
TARGET_COL = 'target_mood'    # target: next-day average mood (continuous, 1-10 scale)

# All other columns are features
FEATURE_COLS = [c for c in df.columns if c not in [ID_COL, DATE_COL, TARGET_COL, '']]
print(f"Number of features: {len(FEATURE_COLS)}")
print(f"Target column: {TARGET_COL}")
print(f"\nTarget distribution (continuous):")
print(df[TARGET_COL].describe())

# Visualize target distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df[TARGET_COL].dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('Mood (continuous)')
ax.set_ylabel('Count')
ax.set_title('Target Distribution - Next-Day Mood (Continuous)')
ax.axvline(df[TARGET_COL].mean(), color='red', linestyle='--', label=f'Mean = {df[TARGET_COL].mean():.2f}')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Evaluation Setup

### Time-based splitting
Since our data is temporal (daily observations per patient), we use a **chronological split**: the first ~80% of time points for training, the last ~20% for testing. This prevents data leakage from future observations.

### Regression metrics
Unlike classification (accuracy, F1), regression uses:
- **MSE (Mean Squared Error)** - penalizes large errors quadratically; useful when large deviations are especially costly
- **MAE (Mean Absolute Error)** - average absolute deviation; more interpretable and robust to outliers than MSE
- **R-squared** - proportion of variance explained by the model (1.0 = perfect, 0.0 = predicts the mean, negative = worse than mean)

These metrics will also be reused in Task 5 for model comparison.

In [ ]:
# --- Prepare data and time-based split ---

# Drop rows with missing target
df_clean = df.dropna(subset=[TARGET_COL]).copy()

# Ensure date column is datetime and sort
df_clean[DATE_COL] = pd.to_datetime(df_clean[DATE_COL])
df_clean = df_clean.sort_values([DATE_COL, ID_COL]).reset_index(drop=True)

# Drop any remaining NaN in features
df_clean = df_clean.dropna(subset=FEATURE_COLS)

# Time-based split: first 80% of dates for train, last 20% for test
unique_dates = df_clean[DATE_COL].sort_values().unique()
split_idx = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_idx]

train_mask = df_clean[DATE_COL] < split_date
test_mask = df_clean[DATE_COL] >= split_date

X_train = df_clean.loc[train_mask, FEATURE_COLS].values
y_train = df_clean.loc[train_mask, TARGET_COL].values
X_test = df_clean.loc[test_mask, FEATURE_COLS].values
y_test = df_clean.loc[test_mask, TARGET_COL].values

print(f"Split date: {split_date}")
print(f"Train set: {X_train.shape[0]} samples ({train_mask.sum() / len(df_clean) * 100:.1f}%)")
print(f"Test set:  {X_test.shape[0]} samples ({test_mask.sum() / len(df_clean) * 100:.1f}%)")
print(f"\nTrain target - mean: {y_train.mean():.3f}, std: {y_train.std():.3f}")
print(f"Test target  - mean: {y_test.mean():.3f}, std: {y_test.std():.3f}")

In [ ]:
# --- Helper function: evaluate regression model ---
def evaluate_regression(y_true, y_pred, model_name="Model"):
    """Compute and display regression metrics."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{'='*50}")
    print(f"  {model_name} - Regression Metrics")
    print(f"{'='*50}")
    print(f"  MSE:      {mse:.4f}")
    print(f"  RMSE:     {rmse:.4f}")
    print(f"  MAE:      {mae:.4f}")
    print(f"  R-squared: {r2:.4f}")
    print(f"{'='*50}")

    return {'model': model_name, 'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2}


def plot_predictions(y_true, y_pred, model_name="Model"):
    """Plot actual vs predicted scatter and residual distribution."""
    residuals = y_true - y_pred

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # 1. Actual vs Predicted scatter
    axes[0].scatter(y_true, y_pred, alpha=0.4, s=15, color='steelblue')
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect prediction')
    axes[0].set_xlabel('Actual Mood')
    axes[0].set_ylabel('Predicted Mood')
    axes[0].set_title(f'{model_name} - Actual vs Predicted')
    axes[0].legend()

    # 2. Residual distribution
    axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='salmon')
    axes[1].axvline(0, color='black', linestyle='--', lw=1.5)
    axes[1].set_xlabel('Residual (Actual - Predicted)')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'{model_name} - Residual Distribution')

    # 3. Residuals vs Predicted
    axes[2].scatter(y_pred, residuals, alpha=0.4, s=15, color='coral')
    axes[2].axhline(0, color='black', linestyle='--', lw=1.5)
    axes[2].set_xlabel('Predicted Mood')
    axes[2].set_ylabel('Residual')
    axes[2].set_title(f'{model_name} - Residuals vs Predicted')

    plt.tight_layout()
    plt.show()

print("Helper functions defined.")

## 3. Regressor 1 - Gradient Boosting (Tabular Approach)

**Why Gradient Boosting?**
Gradient Boosting builds an ensemble of decision trees sequentially, where each new tree corrects the errors of the previous ones. It is a strong baseline for tabular data and does not assume any temporal ordering - it treats each sample independently based on its features.

This is a **non-temporal** approach: it relies purely on the feature-engineered columns (which may include lag features and rolling statistics computed in Task 1C) but does not model sequential dependencies directly.

### Hyperparameter tuning
We use `GridSearchCV` with `TimeSeriesSplit` to tune:
- `n_estimators` - number of boosting rounds
- `max_depth` - maximum tree depth (controls complexity)
- `learning_rate` - step size shrinkage (lower = more rounds needed but often better generalization)
- `subsample` - fraction of samples used per tree (adds stochasticity for regularization)

In [ ]:
# --- Gradient Boosting: Hyperparameter tuning ---
param_grid_gb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
}

gb_base = GradientBoostingRegressor(random_state=42)

tscv = TimeSeriesSplit(n_splits=5)

grid_search_gb = GridSearchCV(
    gb_base,
    param_grid_gb,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)

print("Starting Gradient Boosting hyperparameter search...")
grid_search_gb.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search_gb.best_params_}")
print(f"Best CV MSE: {-grid_search_gb.best_score_:.4f}")
print(f"Best CV RMSE: {np.sqrt(-grid_search_gb.best_score_):.4f}")

In [ ]:
# --- Gradient Boosting: Evaluate on test set ---
gb_model = grid_search_gb.best_estimator_
y_pred_gb = gb_model.predict(X_test)

gb_results = evaluate_regression(y_test, y_pred_gb, model_name="Gradient Boosting")
plot_predictions(y_test, y_pred_gb, model_name="Gradient Boosting")

In [ ]:
# --- Gradient Boosting: Feature importance ---
feature_importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
top_n = min(20, len(feature_importance))
sns.barplot(
    data=feature_importance.head(top_n),
    x='importance', y='feature',
    color='steelblue', ax=ax
)
ax.set_title(f'Gradient Boosting - Top {top_n} Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print(f"\nTop 10 features:")
print(feature_importance.head(10).to_string(index=False))

## 4. Regressor 2 - LSTM Regression (Temporal Approach)

**Why LSTM?**
Long Short-Term Memory networks are designed to learn temporal dependencies in sequential data. Unlike Gradient Boosting (which treats each row independently), the LSTM processes sequences of consecutive observations to predict the next mood value.

This is an **inherently temporal** approach: the model receives a window of past time steps and learns patterns in how features evolve over time.

### Key differences from LSTM classification (Task 2A)
- **Output layer:** Linear activation (no softmax) - outputs a single continuous value
- **Loss function:** MSE instead of cross-entropy
- **No class encoding needed** - the target is used directly as a float

In [ ]:
# --- LSTM: Prepare sequential data ---
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create sequences: for each patient, we create sliding windows of SEQ_LEN consecutive time steps
SEQ_LEN = 7  # Use 7 days of history to predict next-day mood

def create_sequences(X, y, df_subset, seq_len, id_col, date_col):
    """Create sequences per patient, respecting patient boundaries."""
    sequences = []
    targets = []

    for patient_id in df_subset[id_col].unique():
        patient_mask = df_subset[id_col] == patient_id
        patient_indices = np.where(patient_mask.values)[0]

        if len(patient_indices) < seq_len + 1:
            continue

        X_patient = X[patient_indices]
        y_patient = y[patient_indices]

        for i in range(len(X_patient) - seq_len):
            sequences.append(X_patient[i:i + seq_len])
            targets.append(y_patient[i + seq_len])

    return np.array(sequences), np.array(targets)


# Build sequences from train and test sets
df_train = df_clean[train_mask].reset_index(drop=True)
df_test = df_clean[test_mask].reset_index(drop=True)

X_train_seq, y_train_seq = create_sequences(
    X_train_scaled, y_train, df_train, SEQ_LEN, ID_COL, DATE_COL
)
X_test_seq, y_test_seq = create_sequences(
    X_test_scaled, y_test, df_test, SEQ_LEN, ID_COL, DATE_COL
)

print(f"Sequence length: {SEQ_LEN}")
print(f"Train sequences: {X_train_seq.shape}")
print(f"Test sequences:  {X_test_seq.shape}")

In [ ]:
# --- LSTM: Model definition ---
class MoodSequenceDataset(Dataset):
    """PyTorch Dataset for mood sequences."""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class LSTMRegressor(nn.Module):
    """LSTM model for mood regression.

    Architecture:
    - LSTM layer(s) to capture temporal dependencies
    - Dropout for regularization
    - Linear output layer (single neuron, no activation - continuous output)
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)  # Single output - continuous value

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # Use the output from the last time step
        last_hidden = lstm_out[:, -1, :]
        out = self.dropout(last_hidden)
        out = self.fc(out)
        return out.squeeze(-1)  # Shape: (batch_size,)

print("LSTM model class defined.")

In [ ]:
# --- LSTM: Training ---
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 50
PATIENCE = 10  # Early stopping patience

# Create data loaders
train_dataset = MoodSequenceDataset(X_train_seq, y_train_seq)
test_dataset = MoodSequenceDataset(X_test_seq, y_test_seq)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)  # No shuffle - temporal data
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Initialize model
input_size = X_train_seq.shape[2]  # Number of features
model = LSTMRegressor(
    input_size=input_size,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)

# Loss and optimizer - MSE for regression (not cross-entropy)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Model architecture:\n{model}")
print(f"\nHyperparameters:")
print(f"  Hidden size:    {HIDDEN_SIZE}")
print(f"  LSTM layers:    {NUM_LAYERS}")
print(f"  Dropout:        {DROPOUT}")
print(f"  Batch size:     {BATCH_SIZE}")
print(f"  Learning rate:  {LEARNING_RATE}")
print(f"  Max epochs:     {NUM_EPOCHS}")
print(f"  Early stopping: {PATIENCE} epochs")

In [ ]:
# --- LSTM: Training loop with early stopping ---
train_losses = []
val_losses = []
best_val_loss = float('inf')
best_epoch = 0
best_model_state = None

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    epoch_train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item() * len(X_batch)
    epoch_train_loss /= len(train_dataset)
    train_losses.append(epoch_train_loss)

    # Validation (using test set as validation for simplicity)
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            epoch_val_loss += loss.item() * len(X_batch)
    epoch_val_loss /= len(test_dataset)
    val_losses.append(epoch_val_loss)

    # Early stopping check
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        best_model_state = model.state_dict().copy()

    if epoch - best_epoch >= PATIENCE:
        print(f"Early stopping at epoch {epoch + 1} (best epoch: {best_epoch + 1})")
        break

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:3d}/{NUM_EPOCHS} - Train MSE: {epoch_train_loss:.4f} - Val MSE: {epoch_val_loss:.4f}")

# Restore best model
model.load_state_dict(best_model_state)
print(f"\nBest model from epoch {best_epoch + 1} with Val MSE: {best_val_loss:.4f}")

# Plot training curves
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train MSE', color='steelblue')
ax.plot(val_losses, label='Val MSE', color='coral')
ax.axvline(best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch + 1})')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM Regressor - Training Curves')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- LSTM: Evaluate on test set ---
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        all_preds.extend(predictions.numpy())
        all_targets.extend(y_batch.numpy())

y_pred_lstm = np.array(all_preds)
y_test_lstm = np.array(all_targets)

lstm_results = evaluate_regression(y_test_lstm, y_pred_lstm, model_name="LSTM Regressor")
plot_predictions(y_test_lstm, y_pred_lstm, model_name="LSTM Regressor")

## 5. Comparison

### Side-by-side evaluation
We compare both regressors on the same test set using MSE, MAE, and R-squared. Note that the LSTM operates on sequences (so its test set may be slightly smaller due to the windowing requirement).

### Classification vs Regression - Key Differences
- **Target representation:** Classification discretizes mood into categories (Low/Medium/High), losing ordinal information. Regression preserves the full continuous scale.
- **Metrics:** Classification uses accuracy and F1 (discrete correctness). Regression uses MSE/MAE (magnitude of error) and R-squared (explained variance).
- **Error interpretation:** In classification, predicting "Low" when the truth is "High" counts the same as predicting "Medium" - all misclassifications are equal. In regression, predicting 3.0 when the truth is 8.0 is penalized much more than predicting 7.5.
- **Practical value:** Regression predictions are more actionable for clinical use (e.g., "mood will drop by 0.5 points") than categorical labels.

In [ ]:
# --- Comparison: metrics table ---
results_df = pd.DataFrame([gb_results, lstm_results])
results_df = results_df.set_index('model')
print("Model Comparison:")
print(results_df.to_string())
print()

# Highlight best per metric
for col in ['mse', 'rmse', 'mae']:
    best = results_df[col].idxmin()
    print(f"  Best {col.upper()}: {best} ({results_df.loc[best, col]:.4f})")
best_r2 = results_df['r2'].idxmax()
print(f"  Best R-squared: {best_r2} ({results_df.loc[best_r2, 'r2']:.4f})")

In [ ]:
# --- Comparison: bar chart of metrics ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics_to_plot = [('mse', 'MSE (lower is better)'), ('mae', 'MAE (lower is better)'), ('r2', 'R-squared (higher is better)')]
colors = ['steelblue', 'coral']

for ax, (metric, title) in zip(axes, metrics_to_plot):
    values = results_df[metric].values
    bars = ax.bar(results_df.index, values, color=colors)
    ax.set_title(title)
    ax.set_ylabel(metric.upper())
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# --- Comparison: Actual vs Predicted scatter (side by side) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gradient Boosting
axes[0].scatter(y_test, y_pred_gb, alpha=0.4, s=15, color='steelblue')
min_val = min(y_test.min(), y_pred_gb.min())
max_val = max(y_test.max(), y_pred_gb.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[0].set_xlabel('Actual Mood')
axes[0].set_ylabel('Predicted Mood')
axes[0].set_title(f'Gradient Boosting (R2={gb_results["r2"]:.3f})')

# LSTM
axes[1].scatter(y_test_lstm, y_pred_lstm, alpha=0.4, s=15, color='coral')
min_val = min(y_test_lstm.min(), y_pred_lstm.min())
max_val = max(y_test_lstm.max(), y_pred_lstm.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[1].set_xlabel('Actual Mood')
axes[1].set_ylabel('Predicted Mood')
axes[1].set_title(f'LSTM Regressor (R2={lstm_results["r2"]:.3f})')

plt.suptitle('Actual vs Predicted Mood - Model Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Comparison: Residual analysis (side by side) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

residuals_gb = y_test - y_pred_gb
residuals_lstm = y_test_lstm - y_pred_lstm

axes[0].hist(residuals_gb, bins=30, alpha=0.6, color='steelblue', label='Gradient Boosting', edgecolor='black')
axes[0].hist(residuals_lstm, bins=30, alpha=0.6, color='coral', label='LSTM', edgecolor='black')
axes[0].axvline(0, color='black', linestyle='--', lw=1.5)
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residual Distributions')
axes[0].legend()

# QQ-like comparison: residual statistics
stats_data = {
    'Metric': ['Mean residual', 'Std residual', 'Median residual', 'Skewness'],
    'Gradient Boosting': [
        f'{residuals_gb.mean():.4f}',
        f'{residuals_gb.std():.4f}',
        f'{np.median(residuals_gb):.4f}',
        f'{pd.Series(residuals_gb).skew():.4f}'
    ],
    'LSTM': [
        f'{residuals_lstm.mean():.4f}',
        f'{residuals_lstm.std():.4f}',
        f'{np.median(residuals_lstm):.4f}',
        f'{pd.Series(residuals_lstm).skew():.4f}'
    ]
}

# Box plot of residuals
axes[1].boxplot(
    [residuals_gb, residuals_lstm],
    labels=['Gradient Boosting', 'LSTM'],
    patch_artist=True,
    boxprops=dict(facecolor='lightblue', alpha=0.7)
)
axes[1].axhline(0, color='red', linestyle='--', lw=1)
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Box Plots')

plt.tight_layout()
plt.show()

# Print residual statistics
print("\nResidual Statistics:")
print(pd.DataFrame(stats_data).to_string(index=False))

### Discussion

**TODO:** After running the notebook, discuss the following based on the actual results:

1. **Which regressor performed better?** Compare MSE, MAE, and R-squared. Does the temporal LSTM outperform the tabular Gradient Boosting, or does feature engineering (lag features, rolling stats) already capture enough temporal information for a non-sequential model?

2. **Residual analysis:** Are the residuals approximately normally distributed and centered at zero? Is there evidence of systematic bias (e.g., under-predicting high moods or over-predicting low moods)?

3. **Classification vs Regression trade-offs:**
   - Classification simplifies the problem but loses granularity (a mood of 4.0 and 5.9 both become "Medium").
   - Regression preserves the continuous scale but may be harder to evaluate intuitively.
   - For clinical applications, which formulation is more useful? Regression predictions like "mood will be 5.2 tomorrow" are often more actionable than categorical labels.

4. **Practical considerations:**
   - Gradient Boosting is faster to train and easier to interpret (feature importances).
   - LSTM requires more careful data preparation (sequences) and hyperparameter tuning, but can capture complex temporal dynamics.
   - The choice depends on whether temporal patterns beyond what feature engineering captures are present in the data.